# Stage 4C — Initial CatBoost Model and Controlled Sensitive Comparison

This notebook reports a lean CatBoost screening round and one controlled sensitive comparison. The locked Test Set stays closed.

## 0. Stage Objective

State the allowed work before loading results.

In [1]:
from IPython.display import display
display({'stage': 'Stage 4C — Initial CatBoost Model and Controlled Sensitive Comparison', 'test_set': 'locked', 'stage4d_started': False})

{'stage': 'Stage 4C — Initial CatBoost Model and Controlled Sensitive Comparison',
 'test_set': 'locked',
 'stage4d_started': False}

**Interpretation.** Stage 4C builds preliminary CatBoost evidence only. It does not use Test data or start Stage 4D.

## 1. Imports and Configuration

Load small tools and set the execution label.

In [2]:
from pathlib import Path
import json, os
import numpy as np
import pandas as pd
from IPython.display import display
import stage4_catboost_utils as c4
ROOT = Path.cwd().resolve()
CACHE_ONLY = os.environ.get('STAGE4C_CACHE_ONLY', '0') == '1'
display({'root': str(ROOT), 'cache_only': CACHE_ONLY, 'seed': c4.SEED})

{'root': 'D:\\SHARIF\\TERM7\\DATA\\PROJECT\\regresionpart2',
 'cache_only': True,
 'seed': 42}

**Interpretation.** The run uses the project root, seed 42, and an explicit cache-only label.

## 2. Stage 4A and Stage 4B Validation

Confirm that both prerequisite Stages passed.

In [3]:
a = json.loads((ROOT/'artifacts/reports/stage4a_verification.json').read_text(encoding='utf-8'))
b = json.loads((ROOT/'artifacts/reports/stage4b_verification.json').read_text(encoding='utf-8'))
assert a['status'] == b['status'] == 'PASS'
display(pd.DataFrame([{'stage':'Stage 4A','status':a['status']},{'stage':'Stage 4B','status':b['status']}]))

,stage,status
0,Stage 4A,PASS
1,Stage 4B,PASS


**Interpretation.** The CatBoost work starts from verified Stage 4A infrastructure and read-only Stage 4B Feature Packs.

## 3. Protected File Check

Recheck the protected baseline without changing prior files.

In [4]:
protected = c4.recheck_protected(ROOT)
assert protected['status'] == 'PASS'
display({'protected_files': protected['file_count'], 'mismatches': len(protected['mismatches']), 'status': protected['status']})

{'protected_files': 325, 'mismatches': 0, 'status': 'PASS'}

**Interpretation.** All 325 protected files still match their Stage 4C starting hashes.

## 4. Discovery Sample Validation

Check the saved Discovery design before showing model results.

In [5]:
sample = json.loads((ROOT/'artifacts/splits/stage4/stage4_sample_verification.json').read_text(encoding='utf-8'))
d = sample['samples']['discovery']
assert sample['test_overlap_rows'] == 0 and d['valid']
display({'train_rows': d['train_rows'], 'validation_rows': d['validation_rows'], 'test_overlap': sample['test_overlap_rows']})

{'train_rows': 50000, 'validation_rows': 15000, 'test_overlap': 0}

**Interpretation.** The full comparison uses 50,000 Discovery Train rows and 15,000 Discovery Validation rows, all outside Test.

## 5. Lean Screening Subset

Show the smaller deterministic screening subset.

In [6]:
subset = json.loads((ROOT/'artifacts/reports/stage4c_screening_subset_verification.json').read_text(encoding='utf-8'))
assert subset['status'] == 'PASS'
display(pd.Series(subset['rows_by_role'], name='rows').to_frame())

,rows
train,30000
validation,10000


**Interpretation.** Screening used exactly 30,000 Train and 10,000 Validation rows with saved bin labels.

## 6. CatBoost Feature Packs

Load the real Stage 4B packs used by Stage 4C.

In [7]:
packs = json.loads((ROOT/'artifacts/features/stage4/boosting_feature_packs.json').read_text(encoding='utf-8'))['packs']
used = {name: {'numeric':len(packs[name]['numeric']), 'categorical':len(packs[name]['categorical'])} for name in ('boosting_base_v1','catboost_native_v1')}
display(pd.DataFrame(used).T)

,numeric,categorical
boosting_base_v1,11,12
catboost_native_v1,13,22


**Interpretation.** The comparison used only the permitted base and native CatBoost packs. Feature lists came from saved artifacts.

## 7. CPU or GPU Execution Mode

Report the bounded CPU/GPU decision.

In [8]:
display(pd.DataFrame([{'mode':'CPU','seconds':2.8033,'finite':True,'selected':True},{'mode':'GPU','seconds':2.9334,'finite':True,'selected':False}]))

,mode,seconds,finite,selected
0,CPU,2.8033,True,True
1,GPU,2.9334,True,False


**Interpretation.** Both modes passed, but CPU was slightly faster. Stage 4C used CPU with four threads.

## 8. Three Required Candidates

Show the three required non-sensitive Candidate results.

In [9]:
screening = pd.read_csv(ROOT/'artifacts/results/stage4/catboost/initial/catboost_initial_screening.csv')
required = screening.loc[screening['candidate_id'].isin(c4.REQUIRED_CANDIDATES)]
assert len(required) == 3 and required['status'].eq('PASS').all()
display(required[['candidate_id','feature_pack','target_mode','mae','rmse','best_iteration','fit_time_seconds']].sort_values('mae'))

,candidate_id,feature_pack,target_mode,mae,rmse,best_iteration,fit_time_seconds
0,candidate_03_native_log1p,catboost_native_v1,log1p,65.122938,148.344021,995,126.671620
2,candidate_02_native_raw,catboost_native_v1,raw,65.658441,150.029238,999,124.199868
3,candidate_01_base_raw,boosting_base_v1,raw,66.879871,148.178708,998,93.459150


**Interpretation.** Native categories improved MAE over the base pack, and the native log-target Candidate had the best required MAE.

## 9. Optional Local Refinement

Explain the one allowed local refinement.

In [10]:
refine = screening.loc[~screening['candidate_id'].isin(c4.REQUIRED_CANDIDATES)]
assert len(refine) <= 1
display(refine[['candidate_id','depth','l2_leaf_reg','mae','rmse','fit_time_seconds']])

,candidate_id,depth,l2_leaf_reg,mae,rmse,fit_time_seconds
1,candidate_04_safer_refinement,6,20,65.307103,147.604874,78.937247


**Interpretation.** A safer depth and regularization direction reduced RMSE slightly, but it did not improve the primary MAE.

## 10. Candidate Results

Compare all Candidates on original-scale metrics.

In [11]:
display(screening[['selection_rank','candidate_id','target_mode','mae','rmse','rmsle_clipped_zero','top_decile_mae','top_five_percent_mae','selected']].sort_values('selection_rank'))

,selection_rank,candidate_id,target_mode,mae,rmse,rmsle_clipped_zero,top_decile_mae,top_five_percent_mae,selected
0,1,candidate_03_native_log1p,log1p,65.122938,148.344021,0.403206,214.399206,315.483725,True
1,2,candidate_04_safer_refinement,log1p,65.307103,147.604874,0.403677,212.868235,311.417015,False
2,3,candidate_02_native_raw,raw,65.658441,150.029238,0.425728,211.533944,310.000222,False
3,4,candidate_01_base_raw,raw,66.879871,148.178708,0.438307,214.044308,309.383052,False


**Interpretation.** All four allowed Candidates passed. The best validation MAE was 65.123 thousand dollars.

## 11. Preliminary Candidate Selection

Load the preliminary choice made from non-sensitive data only.

In [12]:
frozen = json.loads((ROOT/'artifacts/results/stage4/catboost/initial/catboost_preliminary_configuration.json').read_text(encoding='utf-8'))
assert frozen['selection_source'].startswith('without_sensitive')
display({k:frozen[k] for k in ('selected_candidate_id','feature_pack','target_mode','fixed_iteration_count','selection_reason')})

{'selected_candidate_id': 'candidate_03_native_log1p',
 'feature_pack': 'catboost_native_v1',
 'target_mode': 'log1p',
 'fixed_iteration_count': 995,
 'selection_reason': 'Lowest controlled screening MAE under the tie rules; MAE=65.122938.'}

**Interpretation.** The native log-target Candidate was selected before any sensitive-mode result was available.

## 12. Frozen Configuration

Show the settings frozen for both controlled fits.

In [13]:
display(pd.Series({'feature_pack':frozen['feature_pack'],'target_mode':frozen['target_mode'],'iterations':frozen['fixed_iteration_count'],'seed':frozen['random_seed'],'mode':frozen['execution_mode'],'threads':frozen['thread_count']}))

feature_pack    catboost_native_v1
target_mode                  log1p
iterations                     995
seed                            42
mode                           CPU
threads                          4
dtype: object

**Interpretation.** The pack, target mode, parameters, seed, iteration count, and Pipeline policy were fixed once.

## 13. Controlled Sensitive Comparison

Compare the same frozen configuration on identical Discovery roles.

In [14]:
controlled = pd.read_csv(ROOT/'artifacts/results/stage4/catboost/initial/catboost_controlled_validation_results.csv')
assert controlled['fixed_iteration_count'].nunique() == 1 and len(controlled) == 2
display(controlled[['sensitive_mode','feature_count','mae','rmse','rmsle','r_squared','fit_time_seconds']])

,sensitive_mode,feature_count,mae,rmse,rmsle,r_squared,fit_time_seconds
0,without_sensitive,35,63.541360,138.355852,0.395502,0.664059,157.344741
1,with_sensitive,43,63.505498,139.814777,0.396137,0.656936,183.249057


**Interpretation.** Adding the validated sensitive columns changed MAE by -0.0359. This is an accuracy difference, not a full fairness result.

## 14. Validation Metrics

Show the saved metric differences with a clear sign rule.

In [15]:
comparison = pd.read_csv(ROOT/'artifacts/results/stage4/catboost/initial/catboost_sensitive_comparison.csv')
display(comparison.loc[comparison['metric'].isin(['mae','rmse','rmsle','r_squared','top_decile_mae','top_five_percent_mae'])])

,metric,without_sensitive,with_sensitive,difference_with_minus_without,relative_difference_percent
0,mae,63.541360,63.505498,-0.035862,-0.056439
2,rmse,138.355852,139.814777,1.458925,1.054473
4,r_squared,0.664059,0.656936,-0.007122,-1.072522
5,rmsle,0.395502,0.396137,0.000635,0.160453
11,top_decile_mae,207.394284,206.881676,-0.512608,-0.247166
12,top_five_percent_mae,300.761680,299.155938,-1.605742,-0.533892


**Interpretation.** MAE was 63.541 without sensitive columns and 63.505 with them. RMSE and R² did not improve with sensitive columns.

## 15. Preliminary Model Saving

Verify the two saved preliminary model bundles.

In [16]:
manifest = json.loads((ROOT/'artifacts/manifests/stage4/catboost/catboost_preliminary_model_manifest.json').read_text(encoding='utf-8'))
assert manifest['status'] == 'PASS'
display(pd.DataFrame(manifest['models']).T[['model_path','fixed_iteration_count','reload_status']])

,model_path,fixed_iteration_count,reload_status
without_sensitive,artifacts\models\catboost\preliminary\catboost...,995,PASS
with_sensitive,artifacts\models\catboost\preliminary\catboost...,995,PASS


**Interpretation.** Each mode has its own complete fitted Pipeline and source/sample provenance.

## 16. Clean-Process Reload Tests

Read the separate clean-process reload results.

In [17]:
reloads = pd.read_csv(ROOT/'artifacts/reports/stage4c_reload_verification.csv')
assert reloads['status'].eq('PASS').all()
display(reloads[['sensitive_mode','wall_seconds','status']])

,sensitive_mode,wall_seconds,status
0,without_sensitive,3.162391,PASS
1,with_sensitive,3.365214,PASS


**Interpretation.** Both saved bundles loaded in clean processes and reproduced their saved validation predictions.

## 17. CatBoost Feature Importance

Show CatBoost PredictionValuesChange importance without causal claims.

In [18]:
imp0 = pd.read_csv(ROOT/'artifacts/features/stage4/catboost/catboost_importance_without_sensitive.csv')
imp1 = pd.read_csv(ROOT/'artifacts/features/stage4/catboost/catboost_importance_with_sensitive.csv')
display(imp0.head(10)[['rank','feature','importance','feature_group']]); display(imp1.head(10)[['rank','feature','importance','feature_group']])

,rank,feature,importance,feature_group
0,1,applicant_income_000s,17.355745,original numeric feature
1,2,occupancy_lien_status_group,14.763798,Stage 4B fixed feature
2,3,state_lien_status_group,5.961176,Stage 4B fixed feature
3,4,msamd_name,5.960323,high-cardinality category
4,5,property_purpose_group,5.878638,Stage 4B fixed feature
5,6,loan_type_lien_status_group,4.023560,Stage 4B fixed feature
6,7,applicant_vs_area_income_gap_000s,3.921060,Stage 4B fixed feature
7,8,county_name,3.713982,high-cardinality category
8,9,estimated_tract_family_income_000s,3.274669,Stage 4B fixed feature
9,10,lien_status_name,3.231121,original categorical feature


,rank,feature,importance,feature_group
0,1,occupancy_lien_status_group,16.360594,Stage 4B fixed feature
1,2,applicant_income_000s,12.143983,original numeric feature
2,3,purpose_lien_status_group,6.235879,Stage 4B fixed feature
3,4,state_lien_status_group,5.944045,Stage 4B fixed feature
4,5,lien_status_name,5.617113,original categorical feature
5,6,msamd_name,5.347091,high-cardinality category
6,7,property_purpose_group,5.214315,Stage 4B fixed feature
7,8,county_name,4.270168,high-cardinality category
8,9,loan_type_lien_status_group,3.791113,Stage 4B fixed feature
9,10,estimated_tract_family_income_000s,3.081478,Stage 4B fixed feature


**Interpretation.** Income, lien-related fixed groups, and geography were important for prediction. Importance does not show causality.

## 18. Native CatBoost SHAP

Show bounded native CatBoost SHAP summaries for the same rows.

In [19]:
ids = pd.read_csv(ROOT/'artifacts/manifests/stage4/catboost/catboost_shap_sample_row_ids.csv')
sh0 = pd.read_csv(ROOT/'artifacts/features/stage4/catboost/catboost_shap_importance_without_sensitive.csv')
sh1 = pd.read_csv(ROOT/'artifacts/features/stage4/catboost/catboost_shap_importance_with_sensitive.csv')
assert len(ids) <= 300
display({'shared_rows':len(ids)}); display(sh0.head(10)[['rank','feature','mean_absolute_shap']]); display(sh1.head(10)[['rank','feature','mean_absolute_shap']])

{'shared_rows': 300}

,rank,feature,mean_absolute_shap
0,1,occupancy_lien_status_group,0.137387
1,2,applicant_income_000s,0.120868
2,3,msamd_name,0.090593
3,4,property_purpose_group,0.080643
4,5,state_lien_status_group,0.065837
5,6,estimated_tract_family_income_000s,0.053796
6,7,agency_lien_status_group,0.049521
7,8,county_name,0.044864
8,9,applicant_vs_area_income_gap_000s,0.043637
9,10,purpose_lien_status_group,0.042170


,rank,feature,mean_absolute_shap
0,1,occupancy_lien_status_group,0.132958
1,2,applicant_income_000s,0.128553
2,3,msamd_name,0.091861
3,4,property_purpose_group,0.074976
4,5,state_lien_status_group,0.072004
5,6,county_name,0.052758
6,7,purpose_lien_status_group,0.049898
7,8,estimated_tract_family_income_000s,0.049409
8,9,lien_status_name,0.039532
9,10,applicant_vs_area_income_gap_000s,0.034838


**Interpretation.** The same 300 validation IDs were used in both modes. Native SHAP gives a compact global explanation, not a causal result.

## 19. Error by Target Decile

Measure error across target deciles for both modes.

In [20]:
e0 = pd.read_csv(ROOT/'artifacts/results/stage4/catboost/initial/catboost_error_by_decile_without_sensitive.csv')
e1 = pd.read_csv(ROOT/'artifacts/results/stage4/catboost/initial/catboost_error_by_decile_with_sensitive.csv')
display(e0[['target_decile','rows','mae','mean_signed_error','underestimation_rate']]); display(e1[['target_decile','rows','mae','mean_signed_error','underestimation_rate']])

,target_decile,rows,mae,mean_signed_error,underestimation_rate
0,1,1502,26.179123,21.137597,0.271638
1,2,1517,42.064185,33.748941,0.219512
2,3,1493,39.729220,25.282118,0.332217
3,4,1515,38.356842,17.295124,0.420462
4,5,1485,37.853574,8.349873,0.480135
5,6,1491,46.745751,5.232321,0.547954
6,7,1503,50.078161,-10.768075,0.632069
7,8,1513,63.567437,-19.340967,0.660939
8,9,1483,83.310561,-39.207939,0.702630
9,10,1498,208.048471,-162.618640,0.805741


,target_decile,rows,mae,mean_signed_error,underestimation_rate
0,1,1502,26.243130,21.207012,0.270306
1,2,1517,41.970977,33.628661,0.216216
2,3,1493,39.528774,25.353256,0.334896
3,4,1515,38.142297,17.174283,0.418482
4,5,1485,38.164664,8.369678,0.485522
5,6,1491,46.626478,5.483763,0.537223
6,7,1503,50.575752,-10.424531,0.631404
7,8,1513,63.632139,-19.072332,0.664243
8,9,1483,83.135500,-38.369201,0.703304
9,10,1498,207.555369,-159.731913,0.807744


**Interpretation.** Errors rise strongly in the highest target deciles, and high-value loans are more often underestimated.

## 20. Tail and Worst-Error Analysis

Inspect only compact tail summaries and the largest errors.

In [21]:
tail = pd.read_csv(ROOT/'artifacts/results/stage4/catboost/initial/catboost_tail_error.csv')
display(tail.groupby('sensitive_mode').agg(rows=('row_id','size'), mean_absolute_error=('absolute_error','mean'), maximum_absolute_error=('absolute_error','max')).reset_index()); display(tail[['row_id','y_true','y_pred','absolute_error','sensitive_mode']].head(10))

,sensitive_mode,rows,mean_absolute_error,maximum_absolute_error
0,with_sensitive,20,2236.835221,6983.043937
1,without_sensitive,20,2197.448636,6517.705647


,row_id,y_true,y_pred,absolute_error,sensitive_mode
0,486804,8075.0,1557.294353,6517.705647,without_sensitive
1,314018,6000.0,2009.343125,3990.656875,without_sensitive
2,266497,3650.0,273.794173,3376.205827,without_sensitive
3,486448,4063.0,1310.658185,2752.341815,without_sensitive
4,191772,3950.0,1314.628178,2635.371822,without_sensitive
5,424187,3000.0,648.903076,2351.096924,without_sensitive
6,47294,3500.0,1210.036467,2289.963533,without_sensitive
7,395141,2877.0,968.819861,1908.180139,without_sensitive
8,334063,2645.0,778.086940,1866.913060,without_sensitive
9,82050,3265.0,1408.014208,1856.985792,without_sensitive


**Interpretation.** The worst rows confirm a remaining upper-tail problem. No raw sensitive values are displayed.

## 21. Candidate Features for Stage 4D

List no more than three proposals for independent Stage 4D confirmation.

In [22]:
proposals = pd.read_csv(ROOT/'artifacts/features/stage4/catboost/catboost_round2_feature_candidates.csv')
assert len(proposals) <= 3 and (~proposals['target_derived'].astype(bool)).all() and (~proposals['sensitive_derived'].astype(bool)).all()
display(proposals[['feature_name','formula','evidence','fixed_or_learned']])

,feature_name,formula,evidence,fixed_or_learned
0,applicant_to_estimated_tract_income_ratio,applicant_income_000s / max((hud_median_family...,Income and area-income components had importan...,fixed
1,respondent_purpose_group,respondent_id + ' | ' + loan_purpose_name,Respondent and purpose importance ranks were 1...,learned rare grouping inside Pipeline
2,income_band_lien_status_group,training-fit applicant_income_000s quantile ba...,Applicant income and lien status importance ra...,learned band edges inside Pipeline


**Interpretation.** The proposals use only non-sensitive predictor fields. They are not accepted or implemented in Stage 4C.

## 22. Stage 4C Artifact Summary

Count the main Stage 4C artifacts without showing large tables.

In [23]:
summary = json.loads((ROOT/'artifacts/reports/stage4c_analysis_summary.json').read_text(encoding='utf-8'))
display(pd.Series(summary))

importance_rows      {'without_sensitive': 35, 'with_sensitive': 43}
shap_rows            {'without_sensitive': 35, 'with_sensitive': 43}
shap_sample_rows                                                 300
error_deciles        {'without_sensitive': 10, 'with_sensitive': 10}
feature_proposals                                                  3
registry_rows                                                     12
status                                                          PASS
dtype: object

**Interpretation.** Importance, SHAP, errors, proposals, and Registry exports are present for the required modes.

## 23. Stage 4C Verification

Run the internal pre-review verification gate.

In [24]:
verification = json.loads((ROOT/'artifacts/reports/stage4c_internal_verification.json').read_text(encoding='utf-8'))
assert verification['status'] == 'PASS'
display(pd.Series(verification['checks'], name='pass').to_frame())

,pass
stage4a_pass,True
stage4b_pass,True
protected_hashes_unchanged,True
screening_subset_pass,True
three_required_candidates_complete,True
candidate_budget_at_most_four,True
raw_and_log_targets_evaluated,True
preliminary_configuration_frozen,True
selection_non_sensitive_only,True
two_controlled_results,True


**Interpretation.** Every heavy-artifact and safety check needed before notebook execution passed.

## 24. Stage 4C Completion Note

Close the notebook execution while leaving final review as an external gate.

In [25]:
code_sources = [cell.source for cell in nbformat.read(ROOT/'REGRESSION_PART4_CATBOOST.ipynb', as_version=4).cells if cell.cell_type == 'code'] if False else []
display({'model_work': 'complete', 'test_predictions': 0, 'stage4d_started': False, 'cache_only_run': CACHE_ONLY})

{'model_work': 'complete',
 'test_predictions': 0,
 'stage4d_started': False,
 'cache_only_run': True}

**Interpretation.** The Stage 4C model work is complete. Final PASS also requires the saved execution audit and independent Reviewer report.

## 25. Stage 4C Recovery and Prerequisite Gate

**Reason.** This section records the validated evidence for stage 4c recovery and prerequisite gate before the cached code is displayed.

In [26]:
import stage4de_catboost_utils as de
STAGE4DE_CACHE_ONLY = os.environ.get('STAGE4DE_CACHE_ONLY', '0') == '1'
gate = json.loads((ROOT/'artifacts/reports/stage4c_verification.json').read_text(encoding='utf-8'))
display({'stage4c_status': gate['status'], 'stage4de_cache_only': STAGE4DE_CACHE_ONLY, 'test_set': 'locked'})

{'stage4c_status': 'PASS', 'stage4de_cache_only': True, 'test_set': 'locked'}

**Interpretation.** Stage 4C is finalized and protected before Stage 4D-E evidence is used.

## 26. Stage 4D-E Objective

**Reason.** This section records the validated evidence for stage 4d-e objective before the cached code is displayed.

In [27]:
display({'stage': 'Stage 4D-E - CatBoost Feature Confirmation and Final Model', 'feature_confirmation': 'non-sensitive only', 'tuning_candidates': 3, 'final_modes': 2, 'test_set': 'locked'})

{'stage': 'Stage 4D-E - CatBoost Feature Confirmation and Final Model',
 'feature_confirmation': 'non-sensitive only',
 'tuning_candidates': 3,
 'final_modes': 2,
 'test_set': 'locked'}

**Interpretation.** This stage confirms a small Feature set, tunes exactly three final Candidates, fits both frozen modes, and keeps Test locked.

## 27. Stage 4C Feature Proposal Review

**Reason.** This section records the validated evidence for stage 4c feature proposal review before the cached code is displayed.

In [28]:
proposal_review = pd.read_csv(ROOT/'artifacts/features/stage4/catboost/catboost_round2_proposal_review.csv')
display(proposal_review[['feature_name','target_derived','sensitive_derived','formula_stable','approved_for_combined_confirmation']])

,feature_name,target_derived,sensitive_derived,formula_stable,approved_for_combined_confirmation
0,applicant_to_estimated_tract_income_ratio,False,False,True,True
1,respondent_purpose_group,False,False,True,True
2,income_band_lien_status_group,False,False,True,True


**Interpretation.** All three proposals passed source, target, sensitive, stability, and leakage review before confirmation.

## 28. CatBoost Feature Engineer v2

**Reason.** This section records the validated evidence for catboost feature engineer v2 before the cached code is displayed.

In [29]:
v2 = json.loads((ROOT/'artifacts/manifests/stage4/catboost/catboost_feature_engineer_v2_manifest.json').read_text(encoding='utf-8'))
display({'status': v2['status'], 'fit_rows': v2['fit_rows'], 'validation_rows': v2['validation_rows'], **v2['checks']})

{'status': 'PASS',
 'fit_rows': 4000,
 'validation_rows': 1000,
 'row_order_preserved': True,
 'source_unchanged': True,
 'all_v1_features_preserved': True,
 'all_approved_features_created': True,
 'ratio_finite_or_missing': True,
 'ratio_zero_uses_epsilon_and_flag': True,
 'ratio_missing_source_stays_missing': True,
 'learned_edges_training_fit': True,
 'serialization_roundtrip': True,
 'clean_process_import_transform': True,
 'target_absent': True,
 'sensitive_sources_absent': True}

**Interpretation.** The v2 transformer preserved row order, learned its income bands on training rows, and passed saved and clean-process transform tests.

## 29. Feature Confirmation Sample

**Reason.** This section records the validated evidence for feature confirmation sample before the cached code is displayed.

In [30]:
sample_check = json.loads((ROOT/'artifacts/splits/stage4/stage4_sample_verification.json').read_text(encoding='utf-8'))
display(sample_check['samples']['feature_confirmation'])

{'rows': 100000,
 'train_rows': 80000,
 'validation_rows': 20000,
 'expected_rows': 100000,
 'row_ids_unique': True,
 'maximum_target_bin_proportion_difference': 1.2069997098462792e-05,
 'role_maximum_target_bin_proportion_difference': {'train': 7.06999709847167e-06,
  'validation': 3.2069997098468916e-05},
 'sha256': '2dabbdc5e6b0fee9bc50e63feeb912e3096a5408cafbbf1685c1e00332380cf4',
 'valid': True}

**Interpretation.** Feature confirmation used the saved 80,000 Train and 20,000 Validation rows without sensitive Features or Test overlap.

## 30. Lean Feature Confirmation

**Reason.** This section records the validated evidence for lean feature confirmation before the cached code is displayed.

In [31]:
confirmation = pd.read_csv(ROOT/'artifacts/results/stage4/catboost/feature_confirmation/catboost_feature_confirmation_results.csv')
impact = json.loads((ROOT/'artifacts/results/stage4/catboost/final/catboost_feature_engineering_impact.json').read_text(encoding='utf-8'))
display(confirmation[['fit_id','feature_pack_id','mae','rmse','rmsle','top_decile_mae','p90_absolute_error','fit_time_seconds','selected_final_pack']])
display(impact)

,fit_id,feature_pack_id,mae,rmse,rmsle,top_decile_mae,p90_absolute_error,fit_time_seconds,selected_final_pack
0,confirmation_original,catboost_native_v1,64.439894,137.438978,0.395428,210.339175,136.791030,207.092557,True
1,confirmation_combined_v2,catboost_round2_combined_v2,64.498633,135.299333,0.395984,210.638842,136.699191,226.622873,False
2,confirmation_ratio_rescue_v2,catboost_round2_ratio_rescue_v2,64.483222,135.623711,0.395834,209.765711,137.690811,208.424700,False


{'stage': 'stage4de',
 'combined_accepted': False,
 'combined_acceptance_rule': 'none',
 'combined_impact': {'mae_change_percent': 0.09115439956615255,
  'top_decile_mae_change_percent': 0.14246854163906314,
  'p90_change_percent': -0.06713804269297936,
  'main_rule': False,
  'tail_rule': False,
  'stability_rule': False},
 'rescue_ran': True,
 'rescue_accepted': False,
 'rescue_acceptance_rule': 'none',
 'rescue_impact': {'mae_change_percent': 0.06723857992785755,
  'top_decile_mae_change_percent': -0.27263731843878497,
  'p90_change_percent': 0.6577779718503038,
  'main_rule': False,
  'tail_rule': False,
  'stability_rule': False},
 'selection': 'original_stage4c_pack',
 'final_acceptance_rule': 'retain_original',
 'status': 'PASS'}

**Interpretation.** The combined pack and ratio rescue both missed the fixed acceptance rules. The original pack remained selected.

## 31. Final CatBoost Feature Pack

**Reason.** This section records the validated evidence for final catboost feature pack before the cached code is displayed.

In [32]:
final_pack = json.loads((ROOT/'artifacts/results/stage4/catboost/final/catboost_final_feature_pack.json').read_text(encoding='utf-8'))
display({'selection': final_pack['selection'], 'pack_id': final_pack['feature_pack']['pack_id'], 'selected_proposals': final_pack['selected_proposals'], 'acceptance_rule': final_pack['acceptance_rule']})

{'selection': 'original_stage4c_pack',
 'pack_id': 'catboost_native_v1',
 'selected_proposals': [],
 'acceptance_rule': 'retain_original'}

**Interpretation.** The final pack is the unchanged Stage 4C native CatBoost pack. No proposed Stage 4D Feature was accepted.

## 32. Final Selection Sample

**Reason.** This section records the validated evidence for final selection sample before the cached code is displayed.

In [33]:
display(sample_check['samples']['final_selection'])

{'rows': 125000,
 'train_rows': 100000,
 'validation_rows': 25000,
 'expected_rows': 125000,
 'row_ids_unique': True,
 'maximum_target_bin_proportion_difference': 7.974866679333337e-06,
 'role_maximum_target_bin_proportion_difference': {'train': 5.974866679345214e-06,
  'validation': 1.5974866679341337e-05},
 'sha256': '445a077b3fc1963bec38d8ba90384df2e1400e3897b25f0ecbfce78cb96c49ef',
 'valid': True}

**Interpretation.** Tuning and the controlled comparison used the saved 100,000 Train and 25,000 Validation rows. Test remained locked.

## 33. Three Final Tuning Candidates

**Reason.** This section records the validated evidence for three final tuning candidates before the cached code is displayed.

In [34]:
tuning = pd.read_csv(ROOT/'artifacts/results/stage4/catboost/final/catboost_final_tuning.csv')
display(tuning[['mae_rank','fit_id','depth','learning_rate','l2_leaf_reg','best_iteration','mae','rmse','r_squared','top_decile_mae','fit_time_seconds','selected']])

,mae_rank,fit_id,depth,learning_rate,l2_leaf_reg,best_iteration,mae,rmse,r_squared,top_decile_mae,fit_time_seconds,selected
0,1,tuning_c_flexible,9,0.04,10,1999,63.324544,131.370340,0.688445,203.739052,600.103542,False
1,2,tuning_b_safer,6,0.05,20,2000,63.376185,130.272187,0.693632,202.013002,321.937650,True
2,3,tuning_a_baseline,8,0.05,10,1681,63.437519,132.349827,0.683782,203.855689,429.684738,False


**Interpretation.** Candidate C had the lowest MAE, but Candidate B was within the 0.25 percent tie band and was safer in depth, RMSE, tail error, signed error, and runtime.

## 34. Final CatBoost Configuration

**Reason.** This section records the validated evidence for final catboost configuration before the cached code is displayed.

In [35]:
final_config = json.loads((ROOT/'artifacts/results/stage4/catboost/final/catboost_final_configuration.json').read_text(encoding='utf-8'))
display(pd.Series({'feature_pack': final_config['feature_pack']['pack_id'], 'target_mode': final_config['target_mode'], 'selected_fit_id': final_config['selected_fit_id'], 'iterations': final_config['fixed_iteration_count'], 'depth': final_config['parameters']['depth'], 'learning_rate': final_config['parameters']['learning_rate'], 'l2_leaf_reg': final_config['parameters']['l2_leaf_reg'], 'execution_mode': final_config['execution_mode'], 'threads': final_config['thread_count']}))

feature_pack       catboost_native_v1
target_mode                     log1p
selected_fit_id        tuning_b_safer
iterations                       2000
depth                               6
learning_rate                    0.05
l2_leaf_reg                        20
execution_mode                    CPU
threads                             4
dtype: object

**Interpretation.** The final model uses the original pack, log1p target, depth 6, learning rate 0.05, L2 value 20, and 2,000 trees on CPU with four threads.

## 35. Controlled Sensitive Comparison

**Reason.** This section records the validated evidence for controlled sensitive comparison before the cached code is displayed.

In [36]:
controlled = pd.read_csv(ROOT/'artifacts/results/stage4/catboost/final/catboost_final_validation_results.csv')
display(controlled[['sensitive_mode','feature_pack_id','best_iteration','feature_count','mae','rmse','rmsle','r_squared','mean_signed_error','top_decile_mae','fit_time_seconds']])

,sensitive_mode,feature_pack_id,best_iteration,feature_count,mae,rmse,rmsle,r_squared,mean_signed_error,top_decile_mae,fit_time_seconds
0,without_sensitive,catboost_native_v1,2000,35,63.376185,130.272187,0.390619,0.693632,-8.875774,202.013002,320.629190
1,with_sensitive,catboost_native_v1,2000,43,63.273525,131.685736,0.390349,0.686947,-9.193469,202.112966,404.242927


**Interpretation.** Both modes used the same pack, parameters, and 2,000 trees. The sensitive mode improved MAE slightly but had worse RMSE and R-squared.

## 36. Final Validation Results

**Reason.** This section records the validated evidence for final validation results before the cached code is displayed.

In [37]:
sensitive_comparison = pd.read_csv(ROOT/'artifacts/results/stage4/catboost/final/catboost_final_sensitive_comparison.csv')
display(sensitive_comparison.loc[sensitive_comparison['metric'].isin(['mae','rmse','rmsle','r_squared','mean_signed_error','top_decile_mae','fit_time_seconds'])])

,metric,without_sensitive,with_sensitive,difference_with_minus_without,relative_difference_percent
0,mae,63.376185,63.273525,-0.102661,-0.161986
2,rmse,130.272187,131.685736,1.413549,1.085074
4,r_squared,0.693632,0.686947,-0.006685,-0.963727
5,rmsle,0.390619,0.390349,-0.000270,-0.069033
8,mean_signed_error,-8.875774,-9.193469,-0.317695,-3.579352
11,top_decile_mae,202.013002,202.112966,0.099964,0.049484
13,fit_time_seconds,320.629190,404.242927,83.613737,26.078018


**Interpretation.** The displayed difference is an accuracy comparison, not a fairness conclusion. The non-sensitive model remains the selection reference.

## 37. Full-Training Model Fits

**Reason.** This section records the validated evidence for full-training model fits before the cached code is displayed.

In [38]:
full_manifest = json.loads((ROOT/'artifacts/results/stage4/catboost/final/catboost_full_train_manifest.json').read_text(encoding='utf-8'))
display(pd.DataFrame([{'sensitive_mode': mode, 'training_rows': item['training_row_count'], 'iterations': item['fixed_iteration_count'], 'features': item['feature_count'], 'fit_seconds': item['fit_seconds'], 'reload_status': item['reload_status']} for mode,item in full_manifest['models'].items()]))

,sensitive_mode,training_rows,iterations,features,fit_seconds,reload_status
0,without_sensitive,399788,2000,35,1541.909597,PASS
1,with_sensitive,399788,2000,43,2273.296105,PASS


**Interpretation.** Each final pipeline used all 399,788 saved Train rows in a separate sequential fit. No Test row was loaded.

## 38. Final Model Manifests

**Reason.** This section records the validated evidence for final model manifests before the cached code is displayed.

In [39]:
display(pd.DataFrame([{'sensitive_mode': mode, 'bundle_sha256': item['model_sha256'], 'native_sha256': item['native_model_sha256'], 'source_digest': item['source_hash_digest']} for mode,item in full_manifest['models'].items()]))

,sensitive_mode,bundle_sha256,native_sha256,source_digest
0,without_sensitive,57672c581ce91dfabb83a75ceaf074adaaab1b3ace7f94...,77c8ea039e0b36d9f6240d35819dee13943649073f658c...,e90f7bb49cce5584c7ab250c1db6a107de5cf640c7839f...
1,with_sensitive,78384aecc0b00e61ab5121219735e18d589e72984fc10a...,45635296ef6b73a946526e7a414a7376c19ba1bf4a51d9...,6dc52dca5a8a7196a75213fab4a5a5c0a541f843902194...


**Interpretation.** Each bundle and native model has source, sample, configuration, and file-hash evidence.

## 39. Clean-Process Reload Verification

**Reason.** This section records the validated evidence for clean-process reload verification before the cached code is displayed.

In [40]:
reloads = pd.read_csv(ROOT/'artifacts/reports/stage4de_catboost_reload_verification.csv')
display(reloads[['sensitive_mode','model_bytes','native_model_bytes','wall_seconds','return_code','status']])

,sensitive_mode,model_bytes,native_model_bytes,wall_seconds,return_code,status
0,without_sensitive,88500228,268020268,4.146007,0,PASS
1,with_sensitive,85203569,256768648,3.892434,0,PASS


**Interpretation.** Both complete pipelines loaded in clean processes and reproduced fixed Train-reference predictions.

## 40. Final Feature Importance

**Reason.** This section records the validated evidence for final feature importance before the cached code is displayed.

In [41]:
final_imp0 = pd.read_csv(ROOT/'artifacts/results/stage4/catboost/final/catboost_final_importance_without_sensitive.csv')
final_imp1 = pd.read_csv(ROOT/'artifacts/results/stage4/catboost/final/catboost_final_importance_with_sensitive.csv')
display(final_imp0.head(12))
display(final_imp1.head(12))

,feature,importance,feature_group,sensitive_mode,importance_type,rank
0,applicant_income_000s,25.467780,original_numeric_or_categorical,without_sensitive,PredictionValuesChange,1
1,occupancy_lien_status_group,19.526435,stage4b_engineered,without_sensitive,PredictionValuesChange,2
2,state_lien_status_group,10.150164,stage4b_engineered,without_sensitive,PredictionValuesChange,3
3,property_purpose_group,5.849300,stage4b_engineered,without_sensitive,PredictionValuesChange,4
4,county_name,5.119888,high_cardinality_categorical,without_sensitive,PredictionValuesChange,5
5,msamd_name,3.537253,high_cardinality_categorical,without_sensitive,PredictionValuesChange,6
6,lien_status_name,3.133724,original_numeric_or_categorical,without_sensitive,PredictionValuesChange,7
7,agency_lien_status_group,2.985477,stage4b_engineered,without_sensitive,PredictionValuesChange,8
8,tract_income_level,2.690770,original_numeric_or_categorical,without_sensitive,PredictionValuesChange,9
9,purpose_lien_status_group,2.668437,stage4b_engineered,without_sensitive,PredictionValuesChange,10


,feature,importance,feature_group,sensitive_mode,importance_type,rank
0,applicant_income_000s,25.358191,original_numeric_or_categorical,with_sensitive,PredictionValuesChange,1
1,occupancy_lien_status_group,20.163415,stage4b_engineered,with_sensitive,PredictionValuesChange,2
2,state_lien_status_group,9.879172,stage4b_engineered,with_sensitive,PredictionValuesChange,3
3,property_purpose_group,5.835992,stage4b_engineered,with_sensitive,PredictionValuesChange,4
4,county_name,5.169710,high_cardinality_categorical,with_sensitive,PredictionValuesChange,5
5,purpose_lien_status_group,4.520907,stage4b_engineered,with_sensitive,PredictionValuesChange,6
6,msamd_name,3.417220,high_cardinality_categorical,with_sensitive,PredictionValuesChange,7
7,agency_lien_status_group,3.241405,stage4b_engineered,with_sensitive,PredictionValuesChange,8
8,estimated_tract_family_income_000s,2.228394,stage4b_engineered,with_sensitive,PredictionValuesChange,9
9,applicant_income_area_group,1.887768,original_numeric_or_categorical,with_sensitive,PredictionValuesChange,10


**Interpretation.** PredictionValuesChange importance is reported for every final model Feature and grouped by Feature origin.

## 41. Final Native SHAP

**Reason.** This section records the validated evidence for final native shap before the cached code is displayed.

In [42]:
shap_ids = pd.read_csv(ROOT/'artifacts/results/stage4/catboost/final/catboost_final_shap_sample_ids.csv')
final_shap0 = pd.read_csv(ROOT/'artifacts/results/stage4/catboost/final/catboost_final_shap_without_sensitive.csv')
final_shap1 = pd.read_csv(ROOT/'artifacts/results/stage4/catboost/final/catboost_final_shap_with_sensitive.csv')
display({'rows': len(shap_ids), 'unique_ids': shap_ids['row_id'].nunique(), 'same_ids_both_modes': True, 'space': 'log1p target'})
display(final_shap0.head(12))
display(final_shap1.head(12))

{'rows': 300,
 'unique_ids': 300,
 'same_ids_both_modes': True,
 'space': 'log1p target'}

,feature,mean_absolute_shap,mean_signed_shap,feature_group,sensitive_mode,shap_space,sample_rows,rank
0,applicant_income_000s,0.158421,-0.007221,original_numeric_or_categorical,without_sensitive,log1p_target,300,1
1,occupancy_lien_status_group,0.129275,0.012786,stage4b_engineered,without_sensitive,log1p_target,300,2
2,state_lien_status_group,0.106637,0.002075,stage4b_engineered,without_sensitive,log1p_target,300,3
3,county_name,0.075835,0.007376,high_cardinality_categorical,without_sensitive,log1p_target,300,4
4,property_purpose_group,0.073887,0.008805,stage4b_engineered,without_sensitive,log1p_target,300,5
5,msamd_name,0.060652,0.002023,high_cardinality_categorical,without_sensitive,log1p_target,300,6
6,agency_lien_status_group,0.041876,0.010086,stage4b_engineered,without_sensitive,log1p_target,300,7
7,tract_income_level,0.034924,0.003475,original_numeric_or_categorical,without_sensitive,log1p_target,300,8
8,estimated_tract_family_income_000s,0.033747,0.001213,stage4b_engineered,without_sensitive,log1p_target,300,9
9,respondent_id,0.031216,0.002643,high_cardinality_categorical,without_sensitive,log1p_target,300,10


,feature,mean_absolute_shap,mean_signed_shap,feature_group,sensitive_mode,shap_space,sample_rows,rank
0,applicant_income_000s,0.158729,0.002777,original_numeric_or_categorical,with_sensitive,log1p_target,300,1
1,occupancy_lien_status_group,0.142117,0.009638,stage4b_engineered,with_sensitive,log1p_target,300,2
2,state_lien_status_group,0.110973,0.004095,stage4b_engineered,with_sensitive,log1p_target,300,3
3,property_purpose_group,0.085110,0.007908,stage4b_engineered,with_sensitive,log1p_target,300,4
4,county_name,0.069306,0.007679,high_cardinality_categorical,with_sensitive,log1p_target,300,5
5,msamd_name,0.064125,-0.001481,high_cardinality_categorical,with_sensitive,log1p_target,300,6
6,estimated_tract_family_income_000s,0.039920,0.001079,stage4b_engineered,with_sensitive,log1p_target,300,7
7,agency_lien_status_group,0.033630,0.008985,stage4b_engineered,with_sensitive,log1p_target,300,8
8,census_tract_number,0.031990,0.000495,high_cardinality_categorical,with_sensitive,log1p_target,300,9
9,respondent_id,0.029150,-0.001415,high_cardinality_categorical,with_sensitive,log1p_target,300,10


**Interpretation.** Both modes use the same 300 Final Selection validation IDs. Values summarize log1p-target contributions and are not raw-dollar effects.

## 42. Previous-Stage Reference

**Reason.** This section records the validated evidence for previous-stage reference before the cached code is displayed.

In [43]:
previous = pd.read_csv(ROOT/'artifacts/results/stage4/catboost/final/catboost_previous_stage_reference.csv')
display(previous)
display('The locked Test Set will later provide the common final comparison.')

,stage,model_name,sensitive_mode,evaluation_scheme,evaluation_rows,mae,rmse,rmsle,r_squared,runtime_seconds,prediction_runtime_seconds,directly_comparable_across_all_rows,comparison_note,locked_test_note
0,Stage 2,lasso,without_sensitive,saved-fold OOF,399788,72.230441,245.237341,0.441602,0.404243,65.342005,1.740441,False,"OOF, Discovery validation, and Final Selection...",The locked Test Set will later provide the com...
1,Stage 3,hist_gradient_boosting,without_sensitive,saved-fold OOF,399788,64.275669,249.121627,0.395075,0.385222,40.657902,9.120964,False,"OOF, Discovery validation, and Final Selection...",The locked Test Set will later provide the com...
2,Stage 4C,catboost,without_sensitive,Discovery validation,15000,63.541360,138.355852,0.395502,0.664059,157.344741,0.227218,False,"OOF, Discovery validation, and Final Selection...",The locked Test Set will later provide the com...
3,Stage 4D-E,catboost,without_sensitive,Final Selection validation,25000,63.376185,130.272187,0.390619,0.693632,320.629190,0.384864,False,"OOF, Discovery validation, and Final Selection...",The locked Test Set will later provide the com...


'The locked Test Set will later provide the common final comparison.'

**Interpretation.** Stage 2 and 3 OOF scores, Stage 4C Discovery scores, and Stage 4D-E Final Selection scores use different rows, so the table is a reference rather than a direct leaderboard.

## 43. Stage 4D-E Artifact Summary

**Reason.** This section records the validated evidence for stage 4d-e artifact summary before the cached code is displayed.

In [44]:
stage4de_summary = json.loads((ROOT/'artifacts/reports/stage4de_analysis_summary.json').read_text(encoding='utf-8'))
runtime = json.loads((ROOT/'artifacts/reports/stage4de_runtime_report.json').read_text(encoding='utf-8'))
display(stage4de_summary)
display(runtime)

{'stage': 'stage4de',
 'stage_name': 'Stage 4D–E — CatBoost Feature Confirmation and Final Model',
 'feature_confirmation_rows': 3,
 'accepted_stage4d_proposals': [],
 'feature_confirmation_selection': 'original_stage4c_pack',
 'tuning_rows': 3,
 'selected_tuning_fit': 'tuning_b_safer',
 'fixed_iteration_count': 2000,
 'controlled_validation_mae': {'without_sensitive': 63.37618549044577,
  'with_sensitive': 63.2735246360925},
 'full_train_rows_per_mode': 399788,
 'registry_rows': 16,
 'runtime_accounted_seconds': 6551.961259199976,
 'next_stage': 'Stage 4F',
 'test_set': 'locked',
 'status': 'PASS'}

{'stage': 'stage4de',
 'fit_count': 10,
 'stage4c_recovery_seconds_reference': 11.13007810000272,
 'feature_confirmation_fit_seconds': 642.1401297999982,
 'final_tuning_fit_seconds': 1351.7259299000143,
 'controlled_comparison_fit_seconds': 724.8721173999947,
 'full_train_fit_seconds': 3815.2057021000073,
 'fit_seconds_sum': 6533.943879200015,
 'interpretation_seconds_sum': 4.633921799977543,
 'notebook_execution_seconds': 13.383458199983579,
 'accounted_seconds': 6551.961259199976,
 'budget_seconds': 12600,
 'measurement_scope': 'Sum of Stage 4D-E fit, interpretation, and notebook execution times. Stage 4C recovery is reported separately.',
 'within_budget': True,
 'status': 'PASS'}

**Interpretation.** All planned model, interpretation, Registry, and runtime artifacts are present and the accounted runtime is within 210 minutes.

## 44. Stage 4D-E Verification

**Reason.** This section records the validated evidence for stage 4d-e verification before the cached code is displayed.

In [45]:
internal = json.loads((ROOT/'artifacts/reports/stage4de_internal_verification.json').read_text(encoding='utf-8'))
display(pd.Series(internal['checks'], name='pass'))
display({'status': internal['status']})

stage4c_pass                            True
protected_inputs_unchanged              True
confirmation_fit_count_at_most_three    True
tuning_fit_count_exactly_three          True
controlled_modes_exactly_two            True
full_train_modes_pass                   True
full_train_rows_exact                   True
clean_reload_pass                       True
importance_and_shap_pass                True
registry_rows_unique                    True
runtime_within_budget                   True
required_artifacts_present              True
Name: pass, dtype: bool

{'status': 'PASS'}

**Interpretation.** The pre-notebook safety and completeness checks pass. Final PASS also requires clean notebook runs and independent review.

## 45. CatBoost Track Completion Note

**Reason.** This section records the validated evidence for catboost track completion note before the cached code is displayed.

In [46]:
stage4de_rows = pd.read_csv(ROOT/'artifacts/results/stage4/catboost/final/stage4de_registry_rows.csv')
display({'model_work': 'complete', 'registry_rows': len(stage4de_rows), 'unique_experiment_ids': stage4de_rows['experiment_id'].nunique(), 'test_predictions': 0, 'next_stage_after_final_pass': 'Stage 4F'})

{'model_work': 'complete',
 'registry_rows': 16,
 'unique_experiment_ids': 16,
 'test_predictions': 0,
 'next_stage_after_final_pass': 'Stage 4F'}

**Interpretation.** Model work is complete with Test still locked. Stage 4F may start only after notebook, review, and final verification evidence pass.